In [6]:
import os.path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor  # 导入GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
save_model=True
del_0qf=True #是否删除0屈服强度数据


Regressor=GradientBoostingRegressor
label='qf'  # 问题名称
model_name1 = 'gboost' #  模型名称记录
index_dir='index'  # 索引名称

main_path=f'models/{label}'

index_excel=f'{index_dir}/qf_index.xlsx'
if not os.path.exists(index_dir):
    print('创建index文件夹')
    os.mkdir(index_dir)
else:
    print('index文件夹已存在')
if not os.path.exists(index_excel):
    print('生成index文件')
    df=pd.DataFrame()
    df.to_excel(index_excel)
else:
    print('index文件已存在')
create_dir(main_path,is_mainpath=True)
n_splits=5
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))
# 设置随机种子以确保结果的可重复性
np.random.seed(200)

params={'alpha': 0.9,
 'ccp_alpha': 0.0,
 'criterion': 'friedman_mse',
 'init': None,
 'learning_rate': 0.1,
 'loss': 'squared_error',
 'max_depth': 3,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'n_estimators': 100,
 'n_iter_no_change': None,
 'random_state': 200,
 'subsample': 1.0,
 'tol': 0.0001,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}


df = pd.read_excel('train_set_new.xlsx', index_col=0)

X = df.drop(columns=['Precipitate Distribution', 'Habit Plane', '屈服强度', '抗拉强度 (UTS)', '追踪编号'])
y = df['屈服强度']
# print('X.columns:\n',X.columns)
# print('特征的形状:\n',len(feature_names))
# print('输入数据的形状：\n',X.shape)

# 将目标变量分箱为10个类别有利于进一步分折（但是预测标签不会这个改变）
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)

# 初始化StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=200)

model= Regressor(**params)  # 如果这里固定初始化模型删除掉的话可以在下面循环呢内添加随机初始化或者指定个别参数

# 初始化列表以存储每个折叠的分数
r2_scores = []
mape_scores = []
mae_scores = []
model_scores = []  # 存储每个分折的评估指标

index = 0
num_each_fold_train = np.floor((np.int32(X.shape[0]) - 1) * (n_splits - 1) / n_splits)
num_each_fold_test = (np.int32(X.shape[0]) - 1) // n_splits
for train_index, test_index in skf.split(X, y_binned):
    X_train, X_test = X.iloc[train_index[:int(num_each_fold_train)]], X.iloc[test_index[:int(num_each_fold_test)]]
    y_train, y_test = y.iloc[train_index[:int(num_each_fold_train)]], y.iloc[test_index[:int(num_each_fold_test)]]
    index = index + 1
    fold_dict['fold' + str(index)] = train_index[:int(num_each_fold_train)].astype(
        int)  # 记录训练集索引    # print(f'fold_{index}',fold_dict)
    # 训练模型
    model.fit(X_train, y_train)
    # joblib.dump(model,   f'{main_path}/{model_name1}{index}_new.pkl')
    # print('保存模型文件')

    # 在测试集上进行预测
    y_pred = model.predict(X_test)

    # 计算并存储分数
    r2 = r2_score(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2_scores.append(r2)
    mape_scores.append(mape)
    mae_scores.append(mae)

    # 存储每个分折的评估指标到字典中
    model_scores.append({
        "Fold": index,
        "R^2": r2,
        "MAPE": mape,
        "MAE": mae
    })

# ls = list(fold_dict.values())
# # print('ls',ls)
# df = pd.DataFrame(ls)
# 
# book = load_workbook(index_excel)
# with pd.ExcelWriter(index_excel, engine='openpyxl') as writer:
#     writer.book = book
#     df.to_excel(writer, sheet_name='k-fold')

# 计算最终模型的性能
final_r2 = np.mean(r2_scores)
final_mape = np.mean(mape_scores)
final_mae = np.mean(mae_scores)

# 添加最终模型性能到DataFrame中
model_scores.append({
    "Fold": "Final",
    "R^2": final_r2,
    "MAPE": final_mape,
    "MAE": final_mae
})
print(model_scores)

print('r2:',final_r2,'mae:',final_mae,'mape:',final_mape)
# 创建DataFrame来存储每个分折的评估指标和最终模型性能
# model_scores_df = pd.DataFrame(model_scores)
# 
# save_path = "individual_model_scores.xlsx"
# if not os.path.exists(save_path):
#     df = pd.DataFrame()
#     df.to_excel(save_path, index=False)
# from openpyxl import load_workbook
# from openpyxl.utils.dataframe import dataframe_to_rows
# # 
# def write_to_existing_excel(scores_df, file_path, sheet_name):
#     try:
#         wb = load_workbook(file_path)
#         ws = wb[sheet_name]
#         # 计算新数据的起始行
#         start_row = ws.max_row + 1
#         # 将数据写入工作表
#         for row in dataframe_to_rows(scores_df, index=False, header=True):
#             ws.append(row)
#         # 保存工作表
#         wb.save(file_path)
#     except KeyError:
#         # 如果文件不存在，则创建新文件并将数据写入
#         scores_df.to_excel(file_path, index=False, sheet_name=sheet_name)
# 
# write_to_existing_excel(model_scores_df, save_path, 'qf_gboost')

index文件夹已存在
index文件已存在
文件夹models/qf已存在
[{'Fold': 1, 'R^2': 0.6588094316555602, 'MAPE': 0.3065101245362307, 'MAE': 36.58540581970521}, {'Fold': 2, 'R^2': 0.6808273507253702, 'MAPE': 0.21887335070614627, 'MAE': 35.51607938932206}, {'Fold': 3, 'R^2': 0.5933774970583203, 'MAPE': 0.24288863042903341, 'MAE': 39.612207402485154}, {'Fold': 4, 'R^2': 0.6805475763737989, 'MAPE': 0.22133156201610252, 'MAE': 37.28544788344312}, {'Fold': 5, 'R^2': 0.6732265883877893, 'MAPE': 0.2245217231136244, 'MAE': 35.44991741562237}, {'Fold': 'Final', 'R^2': 0.6573576888401678, 'MAPE': 0.24282507816022747, 'MAE': 36.88981158211558}]
r2: 0.6573576888401678 mae: 36.88981158211558 mape: 0.24282507816022747


In [7]:
import os.path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor  # 导入GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
save_model=True
del_0qf=True #是否删除0屈服强度数据


Regressor=GradientBoostingRegressor
label='qf'  # 问题名称
model_name1 = 'gboost' #  模型名称记录
index_dir='index'  # 索引名称

main_path=f'models/{label}'

index_excel=f'{index_dir}/qf_index.xlsx'
if not os.path.exists(index_dir):
    print('创建index文件夹')
    os.mkdir(index_dir)
else:
    print('index文件夹已存在')
if not os.path.exists(index_excel):
    print('生成index文件')
    df=pd.DataFrame()
    df.to_excel(index_excel)
else:
    print('index文件已存在')
create_dir(main_path,is_mainpath=True)
n_splits=5
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))
# 设置随机种子以确保结果的可重复性
np.random.seed(200)

params={'alpha': 0.9,
 'ccp_alpha': 0.0,
 'criterion': 'friedman_mse',
 'init': None,
 'learning_rate': 0.1,
 'loss': 'squared_error',
 'max_depth': 3,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'n_estimators': 100,
 'n_iter_no_change': None,
 'random_state': 200,
 'subsample': 1.0,
 'tol': 0.0001,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}


df = pd.read_excel('train_set_new.xlsx', index_col=0)

X = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号'])
X['Habit Plane'] = X['Habit Plane'].apply(lambda x: 1 if x != 0 else 0)

y = df['屈服强度']
# print('X.columns:\n',X.columns)
# print('特征的形状:\n',len(feature_names))
# print('输入数据的形状：\n',X.shape)

# 将目标变量分箱为10个类别有利于进一步分折（但是预测标签不会这个改变）
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)

# 初始化StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=200)

model= Regressor(**params)  # 如果这里固定初始化模型删除掉的话可以在下面循环呢内添加随机初始化或者指定个别参数

# 初始化列表以存储每个折叠的分数
r2_scores = []
mape_scores = []
mae_scores = []
model_scores = []  # 存储每个分折的评估指标

index = 0
num_each_fold_train = np.floor((np.int32(X.shape[0]) - 1) * (n_splits - 1) / n_splits)
num_each_fold_test = (np.int32(X.shape[0]) - 1) // n_splits
for train_index, test_index in skf.split(X, y_binned):
    X_train, X_test = X.iloc[train_index[:int(num_each_fold_train)]], X.iloc[test_index[:int(num_each_fold_test)]]
    y_train, y_test = y.iloc[train_index[:int(num_each_fold_train)]], y.iloc[test_index[:int(num_each_fold_test)]]
    index = index + 1
    fold_dict['fold' + str(index)] = train_index[:int(num_each_fold_train)].astype(
        int)  # 记录训练集索引    # print(f'fold_{index}',fold_dict)
    # 训练模型
    model.fit(X_train, y_train)
    # joblib.dump(model,   f'{main_path}/{model_name1}{index}_new.pkl')
    # print('保存模型文件')

    # 在测试集上进行预测
    y_pred = model.predict(X_test)

    # 计算并存储分数
    r2 = r2_score(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2_scores.append(r2)
    mape_scores.append(mape)
    mae_scores.append(mae)

    # 存储每个分折的评估指标到字典中
    model_scores.append({
        "Fold": index,
        "R^2": r2,
        "MAPE": mape,
        "MAE": mae
    })

# ls = list(fold_dict.values())
# # print('ls',ls)
# df = pd.DataFrame(ls)
# 
# book = load_workbook(index_excel)
# with pd.ExcelWriter(index_excel, engine='openpyxl') as writer:
#     writer.book = book
#     df.to_excel(writer, sheet_name='k-fold')

# 计算最终模型的性能
final_r2 = np.mean(r2_scores)
final_mape = np.mean(mape_scores)
final_mae = np.mean(mae_scores)

# 添加最终模型性能到DataFrame中
model_scores.append({
    "Fold": "Final",
    "R^2": final_r2,
    "MAPE": final_mape,
    "MAE": final_mae
})
print(model_scores)

print('r2:',final_r2,'mae:',final_mae,'mape:',final_mape)
# 创建DataFrame来存储每个分折的评估指标和最终模型性能
# model_scores_df = pd.DataFrame(model_scores)
# 
# save_path = "individual_model_scores.xlsx"
# if not os.path.exists(save_path):
#     df = pd.DataFrame()
#     df.to_excel(save_path, index=False)
# from openpyxl import load_workbook
# from openpyxl.utils.dataframe import dataframe_to_rows
# # 
# def write_to_existing_excel(scores_df, file_path, sheet_name):
#     try:
#         wb = load_workbook(file_path)
#         ws = wb[sheet_name]
#         # 计算新数据的起始行
#         start_row = ws.max_row + 1
#         # 将数据写入工作表
#         for row in dataframe_to_rows(scores_df, index=False, header=True):
#             ws.append(row)
#         # 保存工作表
#         wb.save(file_path)
#     except KeyError:
#         # 如果文件不存在，则创建新文件并将数据写入
#         scores_df.to_excel(file_path, index=False, sheet_name=sheet_name)
# 
# write_to_existing_excel(model_scores_df, save_path, 'qf_gboost')

index文件夹已存在
index文件已存在
文件夹models/qf已存在
[{'Fold': 1, 'R^2': 0.6749395108252989, 'MAPE': 0.30232640750851236, 'MAE': 36.21247534588399}, {'Fold': 2, 'R^2': 0.6794145396720989, 'MAPE': 0.21682115863405838, 'MAE': 35.59905043673689}, {'Fold': 3, 'R^2': 0.608018300221975, 'MAPE': 0.23705457103714975, 'MAE': 38.620312168885384}, {'Fold': 4, 'R^2': 0.6902131486723084, 'MAPE': 0.21107031929034226, 'MAE': 36.29997047088716}, {'Fold': 5, 'R^2': 0.6808033534419085, 'MAPE': 0.2187940480500323, 'MAE': 35.10323250841566}, {'Fold': 'Final', 'R^2': 0.666677770566718, 'MAPE': 0.23721330090401901, 'MAE': 36.36700818616181}]
r2: 0.666677770566718 mae: 36.36700818616181 mape: 0.23721330090401901


In [23]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_absolute_error
from utils import create_empty_ls, create_dir, mycopyfile, mycopydir
import joblib
from openpyxl import load_workbook
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor  # 导入GradientBoostingRegressor

# 设置参数
save_model = True
del_0qf = True  # 是否删除0屈服强度数据

Regressor = GradientBoostingRegressor
label = 'qf'  # 问题名称
model_name1 = 'gboost'  # 模型名称记录
index_dir = 'index'  # 索引名称
main_path = f'models/test/{label}'
index_excel = f'{index_dir}/qf_index.xlsx'
if not os.path.exists(main_path):
    os.makedirs(main_path)

if not os.path.exists(index_dir):
    print('创建index文件夹')
    os.mkdir(index_dir)
else:
    print('index文件夹已存在')

if not os.path.exists(index_excel):
    print('生成index文件')
    df = pd.DataFrame()
    df.to_excel(index_excel)
else:
    print('index文件已存在')


n_splits = 5
ls1 = ['fold' + str(i) for i in range(1, n_splits + 1)]
fold_dict = dict(zip(ls1, [0 for i in range(0, len(ls1))]))
np.random.seed(200)

params = {
    'alpha': 0.9,
    'ccp_alpha': 0.0,
    'criterion': 'friedman_mse',
    'init': None,
    'learning_rate': 0.1,
    'loss': 'squared_error',
    'max_depth': 3,
    'max_features': None,
    'max_leaf_nodes': None,
    'min_impurity_decrease': 0.0,
    'min_samples_leaf': 1,
    'min_samples_split': 2,
    'min_weight_fraction_leaf': 0.0,
    'n_estimators': 100,
    'n_iter_no_change': None,
    'random_state': 200,
    'subsample': 1.0,
    'tol': 0.0001,
    'validation_fraction': 0.1,
    'verbose': 0,
    'warm_start': False
}

# 读取训练数据
df = pd.read_excel('train_set_new.xlsx', index_col=0)
X = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号'])
X['Habit Plane'] = X['Habit Plane']
X['Habit Plane'] = X['Habit Plane'].apply(lambda x: 1 if x != 0 else 0)
y = df['屈服强度']

# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)

# 初始化StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=200)
model = Regressor(**params)

# 初始化列表以存储每个折叠的分数
r2_scores = []
mape_scores = []
mae_scores = []
model_scores = []

index = 0
num_each_fold_train = np.floor((np.int32(X.shape[0]) - 1) * (n_splits - 1) / n_splits)
num_each_fold_test = (np.int32(X.shape[0]) - 1) // n_splits

# 训练模型
for train_index, test_index in skf.split(X, y_binned):
    X_train, X_test = X.iloc[train_index[:int(num_each_fold_train)]], X.iloc[test_index[:int(num_each_fold_test)]]
    y_train, y_test = y.iloc[train_index[:int(num_each_fold_train)]], y.iloc[test_index[:int(num_each_fold_test)]]
    index = index + 1
    fold_dict['fold' + str(index)] = train_index[:int(num_each_fold_train)].astype(int)

    model.fit(X_train, y_train)
    joblib.dump(model, f'{main_path}/{model_name1}{index}_new.pkl')

    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2_scores.append(r2)
    mape_scores.append(mape)
    mae_scores.append(mae)

    model_scores.append({
        "Fold": index,
        "R^2": r2,
        "MAPE": mape,
        "MAE": mae
    })

final_r2 = np.mean(r2_scores)
final_mape = np.mean(mape_scores)
final_mae = np.mean(mae_scores)
model_scores.append({
    "Fold": "Final",
    "R^2": final_r2,
    "MAPE": final_mape,
    "MAE": final_mae
})

print(model_scores)
print('r2:', final_r2, 'mae:', final_mae, 'mape:', final_mape)

# 读取测试数据
test_df = pd.read_excel('test_set_new.xlsx', index_col=0)
test_X = test_df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号'])
test_y = test_df['屈服强度']
test_X['Habit Plane'] = test_X['Habit Plane'].apply(lambda x: 1 if x != 0 else 0)

# 三类数据
test_X_habit_0 = test_X[test_X['Habit Plane'] == 0]
test_y_habit_0 = test_y[test_X['Habit Plane'] == 0]

test_X_habit_1 = test_X[test_X['Habit Plane'] != 0]
test_y_habit_1 = test_y[test_X['Habit Plane'] != 0]

test_X_habit_1_forced_0 = test_X_habit_1.copy()
test_X_habit_1_forced_0['Habit Plane'] = 0

# 初始化存储结果的字典
results = {
    'habit_0': [],
    'habit_1': [],
    'habit_1_forced_0': []
}

# 对三类数据进行预测并计算性能
for i in range(1, n_splits + 1):
    model = joblib.load(f'{main_path}/{model_name1}{i}_new.pkl')

    y_pred_habit_0 = model.predict(test_X_habit_0)
    r2_habit_0 = r2_score(test_y_habit_0, y_pred_habit_0)
    mae_habit_0 = mean_absolute_error(test_y_habit_0, y_pred_habit_0)
    mape_habit_0 = mean_absolute_percentage_error(test_y_habit_0, y_pred_habit_0)
    results['habit_0'].append((r2_habit_0, mae_habit_0, mape_habit_0))

    y_pred_habit_1 = model.predict(test_X_habit_1)
    r2_habit_1 = r2_score(test_y_habit_1, y_pred_habit_1)
    mae_habit_1 = mean_absolute_error(test_y_habit_1, y_pred_habit_1)
    mape_habit_1 = mean_absolute_percentage_error(test_y_habit_1, y_pred_habit_1)
    results['habit_1'].append((r2_habit_1, mae_habit_1, mape_habit_1))

    y_pred_habit_1_forced_0 = model.predict(test_X_habit_1_forced_0)
    r2_habit_1_forced_0 = r2_score(test_y_habit_1, y_pred_habit_1_forced_0)
    mae_habit_1_forced_0 = mean_absolute_error(test_y_habit_1, y_pred_habit_1_forced_0)
    mape_habit_1_forced_0 = mean_absolute_percentage_error(test_y_habit_1, y_pred_habit_1_forced_0)
    results['habit_1_forced_0'].append((r2_habit_1_forced_0, mae_habit_1_forced_0, mape_habit_1_forced_0))

# 打印结果
for key, value in results.items():
    r2_scores = [x[0] for x in value]
    mae_scores = [x[1] for x in value]
    mape_scores = [x[2] for x in value]
    print(f'{key} - R^2: {np.mean(r2_scores)}, MAE: {np.mean(mae_scores)}, MAPE: {np.mean(mape_scores)}')


index文件夹已存在
index文件已存在
[{'Fold': 1, 'R^2': 0.6749395108252989, 'MAPE': 0.30232640750851236, 'MAE': 36.21247534588399}, {'Fold': 2, 'R^2': 0.6794145396720989, 'MAPE': 0.21682115863405838, 'MAE': 35.59905043673689}, {'Fold': 3, 'R^2': 0.608018300221975, 'MAPE': 0.23705457103714975, 'MAE': 38.620312168885384}, {'Fold': 4, 'R^2': 0.6902131486723084, 'MAPE': 0.21107031929034226, 'MAE': 36.29997047088716}, {'Fold': 5, 'R^2': 0.6808033534419085, 'MAPE': 0.2187940480500323, 'MAE': 35.10323250841566}, {'Fold': 'Final', 'R^2': 0.666677770566718, 'MAPE': 0.23721330090401901, 'MAE': 36.36700818616181}]
r2: 0.666677770566718 mae: 36.36700818616181 mape: 0.23721330090401901
habit_0 - R^2: 0.6266464022829108, MAE: 28.957826384211266, MAPE: 0.17958544473569715
habit_1 - R^2: 0.7727429505103155, MAE: 28.966401889795378, MAPE: 0.2166126401832424
habit_1_forced_0 - R^2: 0.7508030313203926, MAE: 30.832090839386694, MAPE: 0.24120092130162388


In [10]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_absolute_error
from utils import create_empty_ls, create_dir, mycopyfile, mycopydir
import joblib
from openpyxl import load_workbook
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor  # 导入GradientBoostingRegressor

# 设置参数
save_model = True
del_0qf = True  # 是否删除0屈服强度数据

Regressor = GradientBoostingRegressor
label = 'qf'  # 问题名称
model_name1 = 'gboost'  # 模型名称记录
index_dir = 'index'  # 索引名称
main_path = f'models/test/{label}'
index_excel = f'{index_dir}/qf_index.xlsx'
if not os.path.exists(main_path):
    os.makedirs(main_path)

if not os.path.exists(index_dir):
    print('创建index文件夹')
    os.mkdir(index_dir)
else:
    print('index文件夹已存在')

if not os.path.exists(index_excel):
    print('生成index文件')
    df = pd.DataFrame()
    df.to_excel(index_excel)
else:
    print('index文件已存在')


n_splits = 5
ls1 = ['fold' + str(i) for i in range(1, n_splits + 1)]
fold_dict = dict(zip(ls1, [0 for i in range(0, len(ls1))]))
np.random.seed(200)

params = {
    'alpha': 0.9,
    'ccp_alpha': 0.0,
    'criterion': 'friedman_mse',
    'init': None,
    'learning_rate': 0.1,
    'loss': 'squared_error',
    'max_depth': 3,
    'max_features': None,
    'max_leaf_nodes': None,
    'min_impurity_decrease': 0.0,
    'min_samples_leaf': 1,
    'min_samples_split': 2,
    'min_weight_fraction_leaf': 0.0,
    'n_estimators': 100,
    'n_iter_no_change': None,
    'random_state': 200,
    'subsample': 1.0,
    'tol': 0.0001,
    'validation_fraction': 0.1,
    'verbose': 0,
    'warm_start': False
}

# 读取训练数据
df = pd.read_excel('train_set_new.xlsx', index_col=0)
X = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号'])
X['Habit Plane'] = X['Habit Plane']
# X['Habit Plane'] = X['Habit Plane'].apply(lambda x: 1 if x != 0 else 0)
y = df['屈服强度']

# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)

# 初始化StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=200)
model = Regressor(**params)

# 初始化列表以存储每个折叠的分数
r2_scores = []
mape_scores = []
mae_scores = []
model_scores = []

index = 0
num_each_fold_train = np.floor((np.int32(X.shape[0]) - 1) * (n_splits - 1) / n_splits)
num_each_fold_test = (np.int32(X.shape[0]) - 1) // n_splits

# 训练模型
for train_index, test_index in skf.split(X, y_binned):
    X_train, X_test = X.iloc[train_index[:int(num_each_fold_train)]], X.iloc[test_index[:int(num_each_fold_test)]]
    y_train, y_test = y.iloc[train_index[:int(num_each_fold_train)]], y.iloc[test_index[:int(num_each_fold_test)]]
    index = index + 1
    fold_dict['fold' + str(index)] = train_index[:int(num_each_fold_train)].astype(int)

    model.fit(X_train, y_train)
    joblib.dump(model, f'{main_path}/{model_name1}{index}_new.pkl')

    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2_scores.append(r2)
    mape_scores.append(mape)
    mae_scores.append(mae)

    model_scores.append({
        "Fold": index,
        "R^2": r2,
        "MAPE": mape,
        "MAE": mae
    })

final_r2 = np.mean(r2_scores)
final_mape = np.mean(mape_scores)
final_mae = np.mean(mae_scores)
model_scores.append({
    "Fold": "Final",
    "R^2": final_r2,
    "MAPE": final_mape,
    "MAE": final_mae
})

print(model_scores)
print('r2:', final_r2, 'mae:', final_mae, 'mape:', final_mape)

# 读取测试数据
test_df = pd.read_excel('test_set_new.xlsx', index_col=0)
test_X = test_df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号'])
test_y = test_df['屈服强度']
test_X['Habit Plane'] = test_X['Habit Plane'].apply(lambda x: 1 if x != 0 else 0)

# 三类数据
test_X_habit_0 = test_X[test_X['Habit Plane'] == 0]
test_y_habit_0 = test_y[test_X['Habit Plane'] == 0]

test_X_habit_1 = test_X[test_X['Habit Plane'] != 0]
test_y_habit_1 = test_y[test_X['Habit Plane'] != 0]

test_X_habit_1_forced_0 = test_X_habit_1.copy()
test_X_habit_1_forced_0['Habit Plane'] = 0

# 初始化存储结果的字典
results = {
    'habit_0': [],
    'habit_1': [],
    'habit_1_forced_0': []
}

# 对三类数据进行预测并计算性能
for i in range(1, n_splits + 1):
    model = joblib.load(f'{main_path}/{model_name1}{i}_new.pkl')

    y_pred_habit_0 = model.predict(test_X_habit_0)
    r2_habit_0 = r2_score(test_y_habit_0, y_pred_habit_0)
    mae_habit_0 = mean_absolute_error(test_y_habit_0, y_pred_habit_0)
    mape_habit_0 = mean_absolute_percentage_error(test_y_habit_0, y_pred_habit_0)
    results['habit_0'].append((r2_habit_0, mae_habit_0, mape_habit_0))

    y_pred_habit_1 = model.predict(test_X_habit_1)
    r2_habit_1 = r2_score(test_y_habit_1, y_pred_habit_1)
    mae_habit_1 = mean_absolute_error(test_y_habit_1, y_pred_habit_1)
    mape_habit_1 = mean_absolute_percentage_error(test_y_habit_1, y_pred_habit_1)
    results['habit_1'].append((r2_habit_1, mae_habit_1, mape_habit_1))

    y_pred_habit_1_forced_0 = model.predict(test_X_habit_1_forced_0)
    r2_habit_1_forced_0 = r2_score(test_y_habit_1, y_pred_habit_1_forced_0)
    mae_habit_1_forced_0 = mean_absolute_error(test_y_habit_1, y_pred_habit_1_forced_0)
    mape_habit_1_forced_0 = mean_absolute_percentage_error(test_y_habit_1, y_pred_habit_1_forced_0)
    results['habit_1_forced_0'].append((r2_habit_1_forced_0, mae_habit_1_forced_0, mape_habit_1_forced_0))

# 打印结果
for key, value in results.items():
    r2_scores = [x[0] for x in value]
    mae_scores = [x[1] for x in value]
    mape_scores = [x[2] for x in value]
    print(f'{key} - R^2: {np.mean(r2_scores)}, MAE: {np.mean(mae_scores)}, MAPE: {np.mean(mape_scores)}')


index文件夹已存在
index文件已存在
[{'Fold': 1, 'R^2': 0.6929142838810263, 'MAPE': 0.29571717089259647, 'MAE': 34.485787453795474}, {'Fold': 2, 'R^2': 0.7560437145399481, 'MAPE': 0.1971464899393186, 'MAE': 31.911326345964863}, {'Fold': 3, 'R^2': 0.6334557146926423, 'MAPE': 0.22657966512489136, 'MAE': 36.56935872643579}, {'Fold': 4, 'R^2': 0.7222244180820541, 'MAPE': 0.20085719564686333, 'MAE': 34.44885588448101}, {'Fold': 5, 'R^2': 0.674252908229777, 'MAPE': 0.2135153659652887, 'MAE': 34.47385148364742}, {'Fold': 'Final', 'R^2': 0.6957782078850896, 'MAPE': 0.2267631775137917, 'MAE': 34.377835978864915}]
r2: 0.6957782078850896 mae: 34.377835978864915 mape: 0.2267631775137917
habit_0 - R^2: 0.6337572852658413, MAE: 28.973323475031385, MAPE: 0.17767177165221476
habit_1 - R^2: 0.8420737820584699, MAE: 23.892972500161715, MAPE: 0.17159136860245175
habit_1_forced_0 - R^2: 0.8056557849047797, MAE: 27.795455774881155, MAPE: 0.22194368355925542
